#### Verificación de salida a internet

In [0]:
import requests
resp = requests.get("https://www.portalinmobiliario.com", timeout=15)

if resp.status_code == 200:
    print("Conección exitosa!")
else:
    print("Error!")


#### Configuración y constantes 

In [0]:
import re
import time
import random
import logging
from datetime import date
from dataclasses import dataclass, asdict
from typing import Optional

import requests
from bs4 import BeautifulSoup
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

COMUNAS_GRAN_CONCEPCION = [
    "concepcion-biobio",
    "talcahuano-biobio",
    "hualpen-biobio",
    "san-pedro-de-la-paz-biobio",
    "chiguayante-biobio",
    "penco-biobio",
    "tome-biobio",
    "coronel-biobio",
    "hualqui-biobio",
    "lota-biobio",
]

# Solo departamento, igual que el scraper de producción (el resto del
# pipeline no procesa casas)
TIPOS_PROPIEDAD_PRODUCCION = ["departamento"]
OPERACION = "arriendo"

MAX_PAGINAS_POR_BUSQUEDA = 1000
MAX_PAGINAS_VACIAS_CONSECUTIVAS = 10
MAX_PAGINAS_POR_CORRIDA = 200
MAX_MINUTOS_POR_CORRIDA = 30
RESULTADOS_POR_PAGINA = 48
DELAY_MIN = 3.0
DELAY_MAX = 7.0

BASE_URL = "https://www.portalinmobiliario.com"

RE_ID_AVISO = re.compile(r"(MLC-\d+)")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}

SELECTORES = {
    "tarjeta": ["div.ui-search-result__wrapper", "div.andes-card", "li.ui-search-layout__item"],
    "titulo": ["h2.ui-search-item__title", "h3.poly-component__title", "a.poly-component__title"],
    "link": ["a.ui-search-link", "a.poly-component__title"],
    "precio": ["span.andes-money-amount__fraction"],
    "moneda": ["span.andes-money-amount__currency-symbol"],
    "ubicacion": ["span.ui-search-item__location", "span.poly-component__location"],
}

RE_DORMITORIOS = re.compile(r"(\d+)\s*dormitorios?", re.IGNORECASE)
RE_BANOS = re.compile(r"(\d+)\s*ba[ñn]os?", re.IGNORECASE)
RE_M2 = re.compile(r"([\d.,]+)\s*m[²2]\b", re.IGNORECASE)


@dataclass
class Aviso:
    comuna: str
    tipo_propiedad: str
    operacion: str
    id_aviso: Optional[str] = None
    titulo: Optional[str] = None
    precio: Optional[str] = None
    moneda: Optional[str] = None
    ubicacion: Optional[str] = None
    dormitorios: Optional[str] = None
    banos: Optional[str] = None
    superficie_m2: Optional[str] = None
    url: Optional[str] = None

#### Funciones de parsing

In [0]:
def extraer_id_aviso(url):
    if not url:
        return None
    m = RE_ID_AVISO.search(url)
    return m.group(1) if m else None


def _first_match(soup_or_tag, selector_list):
    for sel in selector_list:
        found = soup_or_tag.select(sel)
        if found:
            return found
    return []


def construir_url(tipo_propiedad, comuna, pagina):
    offset = 1 + (pagina - 1) * RESULTADOS_POR_PAGINA
    if pagina == 1:
        return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}"
    return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}/_Desde_{offset}_NoIndex_True"


def obtener_html(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
    except requests.RequestException as e:
        log.warning(f"Error de red en {url}: {e}")
        return None

    if resp.status_code != 200:
        log.warning(f"Status {resp.status_code} en {url}")
        return None

    if "captcha" in resp.text.lower()[:5000]:
        log.warning(f"Posible CAPTCHA detectado en {url}. Deteniendo esta búsqueda.")
        return None

    return resp.text


def extraer_atributo_texto(tag, selector_list):
    encontrados = _first_match(tag, selector_list)
    if encontrados:
        return encontrados[0].get_text(strip=True)
    return None


def parsear_atributos_regex(tarjeta):
    texto_completo = tarjeta.get_text(" ", strip=True)
    resultado = {"dormitorios": None, "banos": None, "superficie_m2": None}

    m = RE_DORMITORIOS.search(texto_completo)
    if m:
        resultado["dormitorios"] = m.group(1)
    m = RE_BANOS.search(texto_completo)
    if m:
        resultado["banos"] = m.group(1)
    m = RE_M2.search(texto_completo)
    if m:
        resultado["superficie_m2"] = m.group(1)

    return resultado


def parsear_pagina(html, comuna, tipo_propiedad):
    soup = BeautifulSoup(html, "lxml")
    tarjetas = _first_match(soup, SELECTORES["tarjeta"])

    if not tarjetas:
        log.info(f"Sin tarjetas encontradas ({comuna}, {tipo_propiedad}).")
        return []

    avisos = []
    for tarjeta in tarjetas:
        titulo = extraer_atributo_texto(tarjeta, SELECTORES["titulo"])
        precio = extraer_atributo_texto(tarjeta, SELECTORES["precio"])
        moneda = extraer_atributo_texto(tarjeta, SELECTORES["moneda"])
        ubicacion = extraer_atributo_texto(tarjeta, SELECTORES["ubicacion"])

        link_tag = _first_match(tarjeta, SELECTORES["link"])
        url = link_tag[0].get("href") if link_tag else None
        atributos = parsear_atributos_regex(tarjeta)

        avisos.append(Aviso(
            comuna=comuna,
            tipo_propiedad=tipo_propiedad,
            operacion=OPERACION,
            id_aviso=extraer_id_aviso(url),
            titulo=titulo,
            precio=precio,
            moneda=moneda,
            ubicacion=ubicacion,
            dormitorios=atributos["dormitorios"],
            banos=atributos["banos"],
            superficie_m2=atributos["superficie_m2"],
            url=url,
        ))
    return avisos


def limpiar_precio(valor):
    if not valor:
        return None
    solo_numeros = re.sub(r"[^\d]", "", valor)
    return float(solo_numeros) if solo_numeros else None

#### IDs ya conocidos

In [0]:
ids_conocidos = set(
    row["id_aviso"] for row in
    spark.sql("SELECT id_aviso FROM gran_concepcion.01_bronce.avisos").collect()
)
log.info(f"{len(ids_conocidos)} avisos ya existen en Bronce.")

#### Loop de scraping, todo en memoria 

In [0]:
avisos_nuevos_dicts = []
total_vistos = 0
total_nuevos = 0
paginas_recorridas = 0
motivo_corte = None
t0 = time.time()
hoy = date.today().isoformat()

for comuna in COMUNAS_GRAN_CONCEPCION:
    if motivo_corte in ("limite_paginas", "limite_tiempo"):
        break

    for tipo in TIPOS_PROPIEDAD_PRODUCCION:
        if motivo_corte in ("limite_paginas", "limite_tiempo"):
            break

        log.info(f"--- Buscando: {OPERACION} de {tipo} en {comuna} ---")
        paginas_vacias_consecutivas = 0
        pagina = 1

        while True:
            if paginas_recorridas >= MAX_PAGINAS_POR_CORRIDA:
                motivo_corte = "limite_paginas"
                break
            if (time.time() - t0) / 60 >= MAX_MINUTOS_POR_CORRIDA:
                motivo_corte = "limite_tiempo"
                break
            if paginas_vacias_consecutivas >= MAX_PAGINAS_VACIAS_CONSECUTIVAS:
                break
            if pagina > MAX_PAGINAS_POR_BUSQUEDA:
                break

            url = construir_url(tipo, comuna, pagina)
            log.info(f"Página {pagina}: {url}")
            html = obtener_html(url)
            paginas_recorridas += 1

            if html is None:
                break

            avisos = parsear_pagina(html, comuna, tipo)
            if not avisos:
                break

            nuevos_en_pagina = 0
            for aviso in avisos:
                total_vistos += 1
                if not aviso.id_aviso or aviso.id_aviso in ids_conocidos:
                    continue

                d = asdict(aviso)
                d["precio"] = limpiar_precio(d["precio"])
                d["first_seen"] = hoy

                avisos_nuevos_dicts.append(d)
                ids_conocidos.add(aviso.id_aviso)
                nuevos_en_pagina += 1
                total_nuevos += 1

            paginas_vacias_consecutivas = 0 if nuevos_en_pagina > 0 else paginas_vacias_consecutivas + 1
            log.info(f"  -> {len(avisos)} vistos, {nuevos_en_pagina} nuevos "
                      f"(vacías: {paginas_vacias_consecutivas}/{MAX_PAGINAS_VACIAS_CONSECUTIVAS})")

            pagina += 1
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

if motivo_corte is None:
    motivo_corte = "paginas_vacias_consecutivas"

log.info(
    f"Corrida completa. Vistos: {total_vistos} | Nuevos: {total_nuevos} | "
    f"Páginas: {paginas_recorridas} | Motivo de corte: {motivo_corte} | "
    f"Duración: {round(time.time() - t0, 1)}s"
)

#### DataFrame de pandas con lo nuevo

In [0]:
df_nuevos = pd.DataFrame(avisos_nuevos_dicts)
print(f"{len(df_nuevos)} avisos nuevos para insertar")
df_nuevos.head()

#### Transformar datos de pandas a spark

In [0]:
spark.createDataFrame(df_nuevos).createOrReplaceTempView("avisos_nuevos_tmp")

#### El MERGE en SQL

In [0]:
%sql
MERGE INTO gran_concepcion.01_bronce.avisos AS avisos
USING avisos_nuevos_tmp AS nuevos
ON avisos.id_aviso = nuevos.id_aviso
WHEN NOT MATCHED THEN INSERT (
    id_aviso, comuna, tipo_propiedad, operacion, titulo, precio,
    moneda, ubicacion, dormitorios, banos, superficie_m2, url,
    first_seen, estado_publicacion, intentos_fallidos_detalle
) VALUES (
    nuevos.id_aviso,
    nuevos.comuna,
    nuevos.tipo_propiedad,
    nuevos.operacion,
    nuevos.titulo,
    CAST(nuevos.precio AS DOUBLE),
    nuevos.moneda,
    nuevos.ubicacion,
    CAST(nuevos.dormitorios AS DOUBLE),
    CAST(nuevos.banos AS BIGINT),
    CAST(nuevos.superficie_m2 AS DOUBLE),
    nuevos.url,
    CAST(nuevos.first_seen AS DATE),
    'activo',
    0
)